In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [3]:
documents = ["https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/01-intro.md", 
             "https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/02-environment.md", 
             "https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/03-rag.md"
            ]

In [4]:
documents_llm = []

for doc in documents:
        documents_llm.append(doc)

len(documents_llm)

3

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc)
print(documents)

https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/01-intro.md
['https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/01-intro.md', 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/02-environment.md', 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/03-rag.md']


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json

user_prompt = json.dumps(documents)

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
result = response.output_parsed

print(result)

questions=['What does the lesson say is the point of agentic RAG, and how is it different from a basic retrieval setup?', 'Why do we need to set up a local environment before starting this part of the course, and what should it include?', 'In this lesson, what are the main building blocks of a standard RAG pipeline?', 'How does retrieval help an LLM answer questions about content it was not trained on?', 'What kinds of problems can happen if the retrieved documents are weak or irrelevant?']


In [13]:
from evaluation_utils import llm_structured

In [14]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What does this course mean by an agentic RAG system, and how is it different from a regular RAG setup?', 'Why does the lesson say you should have your coding environment ready before starting this part of the course?', 'What core pieces make up a simple RAG pipeline in this lesson, from the user query to the final answer?', 'Why is retrieval such an important step in RAG instead of just sending the whole knowledge base to the model?', 'What kinds of problems can show up when building a RAG system, and how do they affect answer quality?']


In [15]:
usage.input_tokens, usage.output_tokens

(261, 128)

In [16]:
# question 1 140

In [17]:
import pandas as pd
ground_truth = pd.read_csv("ground-truth.csv")
df_ground_truth = pd.DataFrame(ground_truth)

In [18]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [19]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [20]:
len(chunks)

295

In [21]:
!uv pip install onnxruntime

Using Python 3.12.7 environment at: /Users/donnabrown/Desktop/llm-zoomcamp-2026-code/.venv
Checked 1 package in 5ms


In [22]:
from minsearch import Index
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(chunks)

def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)

from minsearch import VectorSearch
from minsearch import Index
from embedder import Embedder




from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
# move outside so it's called only once
embed = Embedder()
def vector_search(query, num_results=5):
    query_vector = embed.encode(query)
    return vindex.search(query_vector, num_results=num_results)

In [23]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [24]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [25]:
df = pd.read_csv("ground-truth.csv")
ground_truth = df.to_dict(orient="records")
q = ground_truth[0]["question"]

In [26]:
text_search_results = text_search(q, num_results=5)

In [27]:
text_search_results[0]

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

In [28]:
# question 2 01-agentic-rag/lessons/03-rag.md

In [29]:
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [30]:
from embedder import Embedder

embed = Embedder()

In [31]:
texts = [chunk["content"] for chunk in chunks]

from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i : i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)
X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [32]:
vindex.fit(X, chunks)

In [33]:
vector_search_results = vector_search(q, num_results=5)

In [34]:
vector_search_results[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [35]:
# question 3 01-agentic-rag/lessons/01-intro.md

In [63]:
def compute_relevance_text(q):
    correct_filename = q["filename"]
    results = text_search(query=q["question"])
    relevance = [int(d["filename"] == correct_filename) for d in results]
    return relevance

In [64]:
from tqdm.auto import tqdm


def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [57]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [58]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [73]:
def compute_relevance(q, search_function):
    correct_filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = [int(d["filename"] == correct_filename) for d in results]
    return relevance

In [74]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)
    return relevance_total

In [75]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [76]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [ ]:
# question 4 0.76

In [ ]:
evaluate(
    ground_truth,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]